# Core 2 · Silver
Typed and checked. The **quality gate** runs every rule first; if one fails, nothing is written and silver keeps the previous version. Checks against other domains read their silver tables through shortcuts.

In [ ]:
# Domain: core · owner: Data platform team
# Attach this notebook to the lakehouse "Core" in the workspace "Solvi Core".
DOMAIN = "core"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to the domain name
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named domain + name
    return spark.read.table(f"{domain}{name}")

In [ ]:
SCHEMA = {
 "site": {
  "site_id": "string",
  "municipality": "string",
  "region": "string",
  "country": "string",
  "latitude": "double",
  "longitude": "double",
  "station_id": "string"
 },
 "weather_station": {
  "station_id": "string",
  "name": "string",
  "latitude": "double",
  "longitude": "double"
 },
 "weather_daily": {
  "station_id": "string",
  "date": "date",
  "irradiation_kwh_m2": "double",
  "temp_c": "double",
  "precip_mm": "double",
  "snow_depth_cm": "double"
 },
 "weather_forecast": {
  "station_id": "string",
  "issued_date": "date",
  "date": "date",
  "temp_min_c": "double",
  "temp_max_c": "double",
  "wind_ms": "double",
  "precip_mm": "double"
 },
 "product": {
  "product_code": "string",
  "product_name": "string",
  "category": "string",
  "unit": "string",
  "list_price_nok": "decimal(12,2)",
  "abc_class": "string"
 },
 "customer": {
  "customer_id": "string",
  "customer_name": "string",
  "segment": "string",
  "site_id": "string",
  "lead_source": "string",
  "created_date": "date"
 }
}
KEYS = {"site": ["site_id"], "weather_station": ["station_id"], "weather_daily": ["station_id", "date"], "weather_forecast": ["station_id", "date"], "product": ["product_code"], "customer": ["customer_id"]}
FKS = [["customer", "site_id", "site", "site_id", "core"], ["site", "station_id", "weather_station", "station_id", "core"], ["weather_forecast", "station_id", "weather_station", "station_id", "core"]]      # child, column, parent, column, parent's domain
RANGES = [["weather_daily", "irradiation_kwh_m2 BETWEEN 0 AND 9"], ["weather_forecast", "temp_min_c <= temp_max_c"]]

def clean(name):
    df = read_table(f"bronze{name.replace('_', '')}")
    return df.select(*[F.nullif(F.trim(F.col(c)), F.lit("")).cast(t).alias(c) for c, t in SCHEMA[name].items()])
silver = {n: clean(n) for n in SCHEMA}
def parent(table, dom):
    return silver[table] if dom == DOMAIN else read_shared(dom, f"silver{table.replace('_', '')}")

results = []
for t, keys in KEYS.items():
    df = silver[t]
    results += [(t, "key not null", df.filter(" OR ".join(f"{k} IS NULL" for k in keys)).count()),
                (t, "key unique", df.groupBy(*keys).count().filter("count > 1").count())]
for c, cc, p, pc, dom in FKS:
    orphans = silver[c].filter(F.col(cc).isNotNull()).join(parent(p, dom).select(F.col(pc).alias(cc)), cc, "left_anti").count()
    results.append((c, f"{cc} -> {dom}.{p}", orphans))
for t, cond in RANGES:
    results.append((t, cond, silver[t].filter(f"NOT ({cond})").count()))
dq = (spark.createDataFrame(results, "table string, rule string, failing_rows long")
        .withColumn("passed", F.col("failing_rows") == 0).withColumn("checked_at", F.current_timestamp()))
display(dq.orderBy("passed", "table"))
failed = dq.filter("NOT passed").count()
if failed:
    raise Exception(f"Quality gate failed in {DOMAIN}: {failed} rule(s). Nothing written to silver.")

In [ ]:
for name, df in silver.items():
    write_table(df, f"silver{name.replace('_', '')}")
write_table(dq, "silverdqlog")